# Gaussian Process Regression

Consider the following [data set](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) that has been created in an energy analysis using 12 different building shapes simulated in Ecotect. The buildings differ with respect to the glazing area, the glazing area distribution, and the orientation, amongst other parameters. The dataset contains eight attributes (or features, denoted by X1 to X8) and two responses (denoted by Y1 and Y2). Explore the possibility of modeling the 'heating load' and the 'cooling load' as a single parameter Gaussian process. Discuss your conclusions.

In [3]:
import kagglehub

# Download latest version
kagglepath="elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'eergy-efficiency-dataset' dataset.
Path to dataset files: /kaggle/input/eergy-efficiency-dataset


In [ ]:
import os
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/ENB2012_data.csv")

By "single parameter Gaussian process," we can interpret this as using a Multi-Output Gaussian Process Regression. scikit-learn handles this natively by fitting a single GPR model that predicts both target variables simultaneously, utilizing a shared kernel and optimized hyperparameters.

In [6]:
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("elikplim/eergy-efficiency-dataset")
df2 = pd.read_csv(path + "/ENB2012_data.csv")

Using Colab cache for faster access to the 'eergy-efficiency-dataset' dataset.


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.metrics import mean_squared_error, r2_score

# 1. Prepare the Data
# Assuming df2 is already loaded from your Kaggle cell
X = df2[['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']]
y = df2[['Y1', 'Y2']] # Multi-output target

# 2. Train-Test Split & Scaling
# GPR relies on distances, so feature scaling is strictly required
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Define the Kernel
# We use a Constant Kernel (scales the variance) * RBF (models the smoothness)
# plus a WhiteKernel to account for global noise in the dataset.
kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2)) \
         + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e1))

# 4. Fit the GPR Model
# n_restarts_optimizer ensures we don't get stuck in a local minimum during hyperparameter tuning
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, random_state=42)
print("Fitting the GPR model (this may take a moment)...")
gpr.fit(X_train_scaled, y_train)

# 5. Predictions and Evaluation
y_pred, y_std = gpr.predict(X_test_scaled, return_std=True)

print("\n--- GPR Results ---")
print("Optimized Kernel:", gpr.kernel_)
print(f"Heating Load (Y1) - MSE: {mean_squared_error(y_test['Y1'], y_pred[:, 0]):.2f}, R^2: {r2_score(y_test['Y1'], y_pred[:, 0]):.4f}")
print(f"Cooling Load (Y2) - MSE: {mean_squared_error(y_test['Y2'], y_pred[:, 1]):.2f}, R^2: {r2_score(y_test['Y2'], y_pred[:, 1]):.4f}")

Fitting the GPR model (this may take a moment)...

--- GPR Results ---
Optimized Kernel: 23.2**2 * RBF(length_scale=2.02) + WhiteKernel(noise_level=0.537)
Heating Load (Y1) - MSE: 0.25, R^2: 0.9976
Cooling Load (Y2) - MSE: 1.75, R^2: 0.9811


Performance: GPR generally performs exceptionally well on this specific dataset due to the highly non-linear, smooth relationships between building geometry ($X_1$-$X_8$) and thermal loads. You should expect an $R^2$ score well above $0.95$.

Shared Kernel: By predicting both loads simultaneously, the model assumes that the underlying physical characteristics that drive heating demand also drive cooling demand. The shared length-scale in the RBF kernel optimizes for the overall geometry.

Uncertainty: A key advantage here over neural networks is that y_std provides a confidence interval for every prediction, which is critical in structural engineering to avoid under-designing HVAC systems.

# Linear Regression

Consider the following [data set](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset). This dataset has 2400 samples provides a comprehensive collection of multi-source building environment data designed to support research in green building design, energy efficiency optimization, and indoor comfort prediction using advanced machine learning and deep learning techniques. Explore the possibility of predicting the 'predicted_energy_demand'  using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.

In [ ]:
import kagglehub

# Download latest version
kagglepath="programmer3/green-building-multi-source-environment-dataset" #"ujjwalchowdhury/energy-efficiency-data-set"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

In [ ]:
import os
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/green_building_dataset.csv")
inspector.df=df2

To justify the choice of parameters, the code below dynamically calculates the Pearson Correlation Coefficient between all available features and the target variable. It then filters out features that have a very weak linear relationship, ensuring the Linear Regression model only uses statistically significant inputs.

In [8]:
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("programmer3/green-building-multi-source-environment-dataset")
df2 = pd.read_csv(path + "/green_building_dataset.csv")

Using Colab cache for faster access to the 'green-building-multi-source-environment-dataset' dataset.


In [9]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Data Preparation (assuming df2 is the green building dataset)
df_gb = df2.copy()

# Ensure we only use numeric columns for correlation analysis
numeric_df = df_gb.select_dtypes(include=['float64', 'int64'])

# 2. Feature Selection via Correlation
corr_matrix = numeric_df.corr()
target_corr = corr_matrix['predicted_energy_demand'].sort_values(ascending=False)

print("--- Feature Correlation with Target ---")
print(target_corr)

# Justification Step: Select features with an absolute correlation > 0.15
# This prevents the model from fitting to noise.
correlation_threshold = 0.15
selected_features = target_corr[abs(target_corr) > correlation_threshold].index.tolist()

# Remove the target variable itself from the features list
if 'predicted_energy_demand' in selected_features:
    selected_features.remove('predicted_energy_demand')

print(f"\nSelected Features (Abs Corr > {correlation_threshold}): {selected_features}")

# 3. Train-Test Split
X_lr = numeric_df[selected_features]
y_lr = numeric_df['predicted_energy_demand']

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(X_lr, y_lr, test_size=0.2, random_state=42)

# 4. Fit the Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train_lr)

# 5. Predict and Evaluate
y_pred_lr = lr_model.predict(X_test_lr)

print("\n--- Linear Regression Results ---")
print(f"Mean Squared Error (MSE): {mean_squared_error(y_test_lr, y_pred_lr):.2f}")
print(f"R-squared (R^2): {r2_score(y_test_lr, y_pred_lr):.4f}")

# 6. Coefficient Analysis
coefficients = pd.DataFrame({'Feature': X_lr.columns, 'Coefficient': lr_model.coef_})
print("\n--- Feature Impacts (Coefficients) ---")
print(coefficients.sort_values(by='Coefficient', key=abs, ascending=False))

--- Feature Correlation with Target ---
predicted_energy_demand    1.000000
ventilation_rate           0.728865
electricity_consumption    0.398703
cooling_energy             0.370632
heating_energy             0.271304
equipment_load             0.058766
occupancy                  0.057655
activity_level             0.018522
wind_speed                 0.011333
indoor_humidity            0.007899
outdoor_temperature        0.006786
outdoor_humidity           0.006451
solar_radiation            0.005331
predicted_comfort_index    0.003568
rainfall                  -0.004161
indoor_temperature        -0.008106
indoor_lighting           -0.020631
indoor_noise              -0.024454
co2_concentration         -0.036466
Name: predicted_energy_demand, dtype: float64

Selected Features (Abs Corr > 0.15): ['ventilation_rate', 'electricity_consumption', 'cooling_energy', 'heating_energy']

--- Linear Regression Results ---
Mean Squared Error (MSE): 4.75
R-squared (R^2): 0.9491

--- Feature Impac

Justification of Parameters: In our report, state that features were selected based on a Pearson correlation threshold (e.g., $>0.15$). This mathematically isolates the variables with the strongest linear relationship to energy demand (like ambient temperature or building size), filtering out irrelevant data.

Interpreting Results: Check the $R^2$ value outputted by the code. If it is lower (e.g., $<0.70$), discuss how building energy systems are highly complex and non-linear (as seen in Part 1), meaning a strictly linear model might be too rigid or simplistic to capture all environmental dynamics perfectly.

Feature Coefficients: Look at the printed coefficients. A positive coefficient means that as that feature increases, energy demand increases. A negative coefficient implies it reduces energy demand. You can use these specific numbers to draw conclusions about green building design principles in your final write-up.